# 機械学習力場 (CHGNET)

In [ ]:
from ase.io import read, write
from ase.visualize import view
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from chgnet.model.dynamics import CHGNetCalculator
from chgnet.model.model import CHGNet
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator

In [ ]:
graphite = read("graphite.cif")
graphite = graphite.repeat((6,4,1))
graphite.center(vacuum=20, axis=2)
graphite.positions[:,2] -= graphite.positions[:,2].min() - 0.001

graphite.calc = CHGNetCalculator(CHGNet.load(model_name='0.2.0'))

In [ ]:
# graphiteでz座標が0.1以上の原子のz座標を指定の値にしたAtomsを返す関数
def set_layer_distance(graphite, z):
    graphite_new = graphite.copy()
    graphite_new.positions[:,2] = np.where(graphite.positions[:,2] > 0.1, z, graphite.positions[:,2])
    return graphite_new

In [ ]:
view(graphite)

In [ ]:
# 2.4から6まで0.2刻みのnp.arrayを作成
layer_distances = np.arange(2.4, 6.2, 0.2)
energy_layer = np.array([])
graphite_structures = []
for z in layer_distances:
    graphite_new = set_layer_distance(graphite, z)
    graphite_structures.append(graphite_new)
    graphite_new.calc = CHGNetCalculator(CHGNet.load(model_name='0.2.0'))
    energy_layer = np.append(energy_layer, graphite_new.get_potential_energy())

In [ ]:
fig, ax = plt.subplots()
ax.plot(layer_distances, energy_layer - energy_layer.min(), marker="o")
ax.set_xlabel("Layer distance ($\mathrm{\AA}$)")
ax.set_ylabel("Energy (eV)")

In [ ]:
write("graphite_atoms.xyz", graphite_structures)

In [ ]:
view(graphite_structures)

<div class="alert alert-success">
Q. Calculate the energy by DFT + D3 as a function of layer distance. Compare the difference of energy profiles between CHGNET and CHGNET+D3. 
</div>

### Hint
You can combine two ase calculators as follows:
```
from ase.calculators.mixing import SumCalculator
combined_calc = SumCalculator([CHGNetCalculator(CHGNet.load(model_name='0.2.0')), TorchDFTD3Calculator()])
```